

### Завдання 1: Виклик LLM з базовим промптом

**Мета:** навчитися викликати LLM через LangChain зі звичайним текстовим промптом.

**Що потрібно зробити:**

1. Створіть промпт, який дозволяє отримати інформацію простою мовою на тему "Квантові обчислення". Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

2. Обмежте відповідь до 200 символів і пропишіть в промпті, аби відповідь була короткою (це зекономить вам час і гроші на згенеровані токени).

3. Встановіть своє значення температури на власний розсуд (тут немає правильного чи неправильного значення) і напишіть коментарем, чому ви обрали саме таке значення для цього завдання.

**Вибір моделі:** можна скористатись як моделлю з HuggingFace, так і ChatGPT будь-якої версії, яка вам до вподоби і пасує за прайсингом. В обох випадках потрібно імпортувати відповідний клас з LangChain для виклику LLM за API.

**Мова запитів:** промпти можна писати як українською, так і англійською — орієнтуйтесь на те, де і для чого ви хочете потім використовувати цей проєкт. У розв'язках промпти — українською.

---

**🔐 Як безпечно зберігати і підвантажувати API-ключі**

API-токен потрібно зчитувати з безпечного джерела, а **не хардкодити в ноутбуці**. Якщо хтось отримає доступ до вашого ключа, він буде витрачати токени за ваш рахунок, а вам це не треба :)

Є кілька способів. Перший ми використовували на лекції, ще два для розширення вашого розуміння, як ще це можна зробити і що шлях не лише один. Для виконання цього ДЗ можете використовувати будь-який спосіб підвантаження ключів у ноутбук.

**Спосіб 1: Файл `creds.json` (рекомендований)**

Створіть файл `creds.json` з вашими ключами, завантажте його в Google Colab під час роботи, але **не здавайте** цей файл у ДЗ і **не комітьте** в git.

```python
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["HF_TOKEN"]
```

**Спосіб 2: Google Colab Secrets**

У лівій панелі Colab натисніть іконку 🔑 (Secrets) → "Add new secret" → введіть назву (наприклад, `HF_TOKEN`) та значення ключа → увімкніть тогл доступу для ноутбука.

```python
from google.colab import userdata
api_key = userdata.get("HF_TOKEN")
```

Зручно тим, що ключ зберігається в акаунті і доступний у всіх ваших ноутбуках. Мінус — при кожній новій сесії потрібно перевірити, що доступ увімкнено.

**Спосіб 3: Google AI Studio (для Gemini API)**

Якщо працюєте з моделями Google Gemini, отримати безкоштовний API-ключ можна в [Google AI Studio](https://aistudio.google.com/app/apikey): увійдіть з Google-акаунтом → натисніть "Get API key" → "Create API key". Далі використовуйте ключ через будь-який із способів вище.



In [1]:
! pip install -q langchain-huggingface==1.2.1

In [2]:
import json

with open("creds.json") as f:
  creds = json.load(f)
api_key = creds["HF_TOKEN"]

In [68]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    max_new_tokens=200,
    top_k=10,
    top_p=0.95,
    temperature=0.05,
    repetition_penalty=1.03,
    huggingfacehub_api_token=api_key,
)

chat = ChatHuggingFace(llm=llm)
print(chat.invoke("Поясни українською мовою: Що таке Квантові обчислення (ключові переваги та поточні дослідження в цій галузі)? Відповідь повинна бути короткою до 200 символів! ").content)

Квантові обчислення дозволяють виконувати складні обчислення швидше, ніж традиційні комп'ютери. Основна перевага — паралельне обчислення всіх можливих варіантів. Поточні дослідження фокусуються на створенні більш ефективних квантових алгоритмів та стабільних квантових елементів.


temperature=0.05.
Обране значення близьке до 0 щоб відповідь була точною без зайвого

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [4]:
from langchain_core.prompts import PromptTemplate

In [5]:
prompt = PromptTemplate(
    input_variables=["тема"],
    template="Поясни українською мовою: Що таке {тема} (ключові переваги та поточні дослідження в цій галузі)? Відповідь повинна бути короткою до 200 символів! ?",
)

перевірка роботи промту

In [6]:
print(prompt.format(тема="Баєсівські методи в машинному навчанні"))

Поясни українською мовою: Що таке Баєсівські методи в машинному навчанні (ключові переваги та поточні дослідження в цій галузі)? Відповідь повинна бути короткою до 200 символів! ?


In [7]:
print(chat.invoke(prompt.format(тема="Баєсівські методи в машинному навчанні")).content)

Баєсівські методи в машинному навчанні дозволяють робити точніші прогнози, враховуючи ймовірності. Вони досліджуються для покращення точності відбірку даних та обробки незвідних даних.


In [8]:
print(chat.invoke(prompt.format(тема="Трансформери в машинному навчанні")).content)

Трансформери — це моделі машинного навчання, які використовують імовірнісні трансформаційні змінні для обробки послідовностей даних. Вони швидко стали популярними через можливість обробки дуже довгих рядків і нічутливість до порядку елементів. Поточні дослідження фокусуються на покращенні їх швидкості та ефективності.


In [9]:
print(chat.invoke(prompt.format(тема="Explainable AI")).content)

Explainable AI (XAI) — це технологія, яка дозволяє розуміти, визначати та обґрунтовувати результати роботи AI-моделей. Ключові переваги: прозорість, віраційність та відповідальність. Поточні дослідження фокусуються на розвитку методів для пояснення різних типів моделей.




### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [10]:
!pip install -q langchain_community duckduckgo_search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [11]:
!pip install -U duckduckgo-search ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.9 MB/s eta 0:00:00


In [92]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import load_tools, Tool, AgentExecutor, AgentType, create_react_agent, initialize_agent, create_structured_chat_agent

search = DuckDuckGoSearchRun()

search_tool = Tool(
    name="search",
    func=search.run,
    description="Search tool. Input should be a plain string."
)
tools = [search_tool]


In [93]:
from langchain_classic import hub

prompt = hub.pull("hwchase17/react")

agent = create_structured_chat_agent(chat, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True,
    handle_parsing_errors=True ,
    max_iterations=3)

In [94]:
agent_executor.invoke({'input':"Search for 'AI papers 2024 archive' and summarize 5 of them in Ukrainian."})



> Entering new AgentExecutor chain...
Could not parse LLM output: I need to search for 'AI papers 2024 archive' and then summarize 5 of the papers in Ukrainian. Since I don't have a direct search tool, I will simulate the search process and then provide a hypothetical summary in Ukrainian.

Action: search
Action Input: {'tool_input': 'AI papers 2024 archive'}
Observ
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE Invalid or incomplete responseCould not parse LLM output: Thought: I will proceed with a hypothetical search and summary since the actual search tool did not provide a valid response. I will create a fictional list of AI papers from 2024 and provide a summary in Ukrainian for each.

Action: search
Action Input: {'tool_input': 'AI papers 2024 archive'}
Observ
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE Invalid or incomplete responseCould not parse LLM output

{'input': "Search for 'AI papers 2024 archive' and summarize 5 of them in Ukrainian.",
 'output': 'Agent stopped due to iteration limit or time limit.'}

Не вдалось налаштувати агента



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [115]:
!pip install -qU langchain-google-genai langchain-community langchain-experimental

In [152]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_experimental.utilities import PythonREPL
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

with open("creds.json") as f:
  creds = json.load(f)
api_key = creds["GEMINI_KEY"]

python_repl = PythonREPL()
python_tool = Tool(
    name="python_repl",
    description="Використовуй для розрахунків. Пиши ТІЛЬКИ код.",
    func=python_repl.run
)
search_wrapper = DuckDuckGoSearchAPIWrapper(region="uk-ua", max_results=3)
search = DuckDuckGoSearchRun()
search_tool = Tool(
    name="search",
    func=search_wrapper.run,
    description="Використовуй для пошуку даних про інфляцію та погоду в Бразилії."
)

tools = [python_tool, search_tool]

In [148]:
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    google_api_key=api_key
)


In [149]:
prompt = hub.pull("hwchase17/react")

In [150]:
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True, max_iterations=10
)

In [151]:
task = """Ти — бізнес-аналітик.
Виконай аналіз експорту апельсинів з Бразилії.

Дані: 2022 - 200т, 2023 - 190т, 2024 - 210т, 2025 - 220т.

Кроки:
1. Розрахуй лінійний прогноз на 2026 рік через python_repl (використовуй метод найменших квадратів або темп росту).
2. Знайди через search прогноз інфляції в Бразилії на 2026 рік та дізнайся, чи є там зараз посуха (drought).
3. Якщо інфляція > 5% АБО є посуха — зменш свій прогноз на 10%.
4. Надай фінальну відповідь українською мовою з поясненням кожного кроку.
"""

agent_executor.invoke({"input": task})




> Entering new AgentExecutor chain...


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}